# Contexto del problema

Este capítulo acompaña la pestaña **Contexto del problema** del dashboard: interpreta cada elemento y agrega detalles que no caben en el dashboard (la construcción del objetivo, la comparación de las funciones de pérdida de los dos modelos base y la verificación de la partición).

In [1]:
import os  # Rutas
import sys  # Configuración del intérprete
sys.path.insert(0, os.path.abspath(".."))  # Carpeta entregable2 (paquete tablero)
sys.path.insert(0, os.path.abspath("../.."))  # Raíz del proyecto (paquete src)
sys.path.insert(0, os.path.abspath("../../entregable1"))  # Paquete e1 (código fuente que se muestra)
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
from IPython.display import HTML, Markdown, display  # Incrustar figuras y texto
from tablero import figuras as fg  # Las mismas figuras del dashboard
from tablero.datos import A, M, MOD, FEATURES, curvas, metricas_umbral, rezago_espacial  # Los mismos artefactos del dashboard
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales


def mostrar(fig):  # Muestra una figura del dashboard como HTML interactivo
    """Pone el fondo oscuro de las tarjetas del dashboard y carga Plotly desde internet."""
    fig.update_layout(paper_bgcolor=fg.SUPERFICIE, plot_bgcolor=fg.SUPERFICIE, margin=dict(l=40, r=20, t=50, b=40))  # Fondo oscuro
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False, config={"displaylogo": False})))  # Figura interactiva
k = A["contexto"]["kpi"]  # Indicadores de la portada
pd.Series({"Casos 2010–2024": f"{k['casos']:,}", "Municipios": k["municipios"], "Departamentos": k["departamentos"], "Semanas": k["semanas"],
           "Año récord": f"{k['anio_record']} ({k['casos_record']:,} casos)", "Prevalencia de brote (entrenamiento)": f"{k['prev_tr']:.1%}",
           "Prevalencia de brote (validación 2022)": f"{k['prev_va']:.1%}", "Predictoras": k["p"]}, name="valor").to_frame()

,valor
Casos 2010–2024,"1,261,746"
Municipios,376
Departamentos,32
Semanas,782
Año récord,"2024 (261,382 casos)"
Prevalencia de brote (entrenamiento),10.3%
Prevalencia de brote (validación 2022),12.4%
Predictoras,34


**Interpretación de los indicadores.** El panel reúne 1,26 millones de casos notificados en 376 municipios durante 782 semanas. 2024 es el año récord y queda dentro de la prueba, así que los modelos se evalúan en un régimen más extremo que el de entrenamiento. La clase brote es minoritaria (≈ 10 %), lo que orienta la elección de métricas.

## Pregunta y objetivos

**Pregunta.** ¿Es posible anticipar, con cuatro semanas de margen, si un municipio entrará en brote de dengue usando solo la información disponible en la semana actual?

El dashboard presenta cuatro objetivos: describir la base de datos y su estructura; definir el brote sin información del futuro; explorar la relación de la historia de casos, el clima y El Niño con los brotes; y entrenar dos modelos base lineales comparados con líneas base triviales.

## Casos por año

In [2]:
mostrar(fg.fig_anual(A["contexto"]["anual"]))  # Misma figura de la pestaña Contexto
A["contexto"]["anual"].to_frame("casos").T

ANO,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
casos,129914,25373,45608,108991,87256,81334,84571,21222,38407,104393,66615,43054,55335,108291,261382


**Interpretación.** Las epidemias de 2010, 2013, 2019 y 2023–2024 dominan la serie y coinciden con episodios de El Niño. 2017 y 2018 son años de muy baja transmisión. Esta alternancia es la razón de que la prevalencia del objetivo cambie tanto entre años.

## Detalle: construcción del objetivo con el canal endémico

El dashboard muestra la fórmula; aquí está el código que la implementa (`src/datos.py`). El umbral se calcula de forma vectorizada: para cada semana se toman los 25 valores históricos (5 años × 5 semanas) y su percentil 75.

In [3]:
import inspect  # Código fuente
from src import datos as sd  # Módulo del canal endémico
print(inspect.getsource(sd.canal_endemico))  # Función que calcula el umbral

def canal_endemico(casos, semanas_por_mun):  # Umbral superior del canal endémico (método de cuartiles, Bortman 1999)
    """Q3 de los casos en las mismas semanas (± VENTANA) de los ANIOS_HISTORIA años anteriores, por municipio."""
    n_mun = len(casos) // semanas_por_mun  # Número de municipios
    C = casos.reshape(n_mun, semanas_por_mun)  # Matriz municipios × semanas
    desfases = np.array([-52 * k + d for k in range(1, cf.ANIOS_HISTORIA + 1)  # Misma semana de cada año previo…
                         for d in range(-cf.VENTANA_SEMANAS, cf.VENTANA_SEMANAS + 1)])  # …y sus vecinas (± 2)
    umbral = np.full(C.shape, np.nan)  # Sin historia suficiente queda nulo
    inicio = 52 * cf.ANIOS_HISTORIA + cf.VENTANA_SEMANAS  # Primera semana con 5 años completos de historia
    columnas = np.arange(inicio, semanas_por_mun)  # Semanas con umbral calculable
    indices = columnas[:, None] + desfases[None, :]  # Para cada semana, las 25 semanas históricas de referencia
    umbral[:, column

In [4]:
serie = A["objetivo"]["canal"][76001]  # Cali
mostrar(fg.fig_canal(serie, "Santiago de Cali"))  # Canal endémico
print(f"Semanas de brote en Cali 2015–2021: {int(serie['brote'].sum())} de {len(serie)} ({serie['brote'].mean():.1%})")  # Resumen

Semanas de brote en Cali 2015–2021: 98 de 366 (26.9%)


**Interpretación.** El umbral (línea punteada) sigue la historia de cada municipio: en Cali es alto porque la transmisión es habitual. Un brote no es "muchos casos", sino **más casos de lo normal para esa época del año en ese municipio**. El requisito de al menos 3 casos evita que un solo caso en un municipio sin historia cuente como brote.

## Detalle: las funciones de pérdida de los dos modelos base

La regresión logística minimiza la pérdida logarítmica; la SVM lineal (`LinearSVC`) minimiza la bisagra al cuadrado. Ambas son funciones convexas del margen $z = \tilde y\,(\mathbf w^\top \mathbf x + b)$ con $\tilde y \in \{-1, 1\}$:

$$\ell_{\text{log}}(z) = \ln(1 + e^{-z}), \qquad \ell_{\text{bisagra}^2}(z) = \max(0,\, 1 - z)^2.$$

La diferencia clave: la bisagra es **exactamente cero** para los puntos bien clasificados con margen ($z \ge 1$), así que la SVM solo aprende de los puntos cercanos a la frontera (vectores de soporte); la logística sigue aprendiendo de todos los puntos y produce probabilidades directamente. Por eso la SVM necesita un paso adicional (escalado de Platt) para dar probabilidades.

In [5]:
import plotly.graph_objects as go  # Figura
z = np.linspace(-3, 3, 300)  # Márgenes
fig = go.Figure()  # Comparación de pérdidas
fig.add_scatter(x=z, y=np.log1p(np.exp(-z)), name="Logística: ln(1 + e^(−z))", line=dict(color=fg.AZUL, width=3))  # Logística
fig.add_scatter(x=z, y=np.maximum(0, 1 - z) ** 2, name="Bisagra al cuadrado: max(0, 1 − z)²", line=dict(color=fg.MAGENTA, width=3))  # SVM
fig.add_scatter(x=z, y=(z < 0).astype(float), name="Pérdida 0-1", line=dict(color=fg.GRIS, dash="dot"))  # Referencia
fig.update_layout(xaxis_title="margen z = ỹ (wᵀx + b)", yaxis_title="pérdida", yaxis_range=[0, 5])  # Ejes
mostrar(fg.tema(fig, 380))  # Muestra

**Interpretación.** Las dos pérdidas son sustitutos convexos de la pérdida 0-1 (que no se puede optimizar directamente). Para márgenes negativos (errores) ambas crecen; para márgenes grandes la logística decae suavemente y la bisagra se anula. Con datos tabulares, muchas filas y las mismas variables, ambas suelen encontrar hiperplanos muy parecidos, como se verá en el capítulo de modelos base.

## Detalle: la partición se verifica, no se supone

In [6]:
mostrar(fg.fig_particion(A["contexto"]["linea_particion"]))  # Línea de tiempo
A["contexto"]["particion"]  # Filas y fechas de cada periodo

,filas,desde,hasta
periodo,,,
historia,96256,2010-01-04,2014-11-24
entrenamiento,136112,2014-12-29,2021-11-29
validación,18048,2022-01-03,2022-11-28
prueba,37600,2023-01-02,2024-11-25
purgada,6016,2014-12-01,2024-12-23


**Interpretación.** Las 6 016 filas *purgadas* son las cuatro semanas de cada frontera, cuya etiqueta cae en el periodo siguiente (por ejemplo, diciembre de 2021, cuyo objetivo es enero de 2022). Con la purga, la última etiqueta del entrenamiento es anterior a la primera semana de la validación, y lo mismo entre validación y prueba: ninguna etiqueta se usa fuera de su periodo.

## Detalle: consideraciones éticas

El panel contiene solo conteos agregados; no requiere consentimiento ni comité de ética. Los cruces finos (edad × sexo × semana en municipios pequeños) no se publican para evitar cualquier riesgo de reidentificación. El riesgo más relevante es de **equidad**: si la notificación es peor donde hay menos recursos, el modelo aprenderá menos riesgo justo donde la vigilancia es más débil.